In [ ]:
import torch
from transformers import AutoTokenizer
from arabert.preprocess import ArabertPreprocessor
# for text classification
from transformers import AutoModelForSequenceClassification

## araBERT Preprocessor

In [ ]:
model_name="bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name)

## Initializing Model

In [ ]:
checkpoint = "aubmindlab/bert-base-arabert"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, return_dict=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

## Inference

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

In [ ]:
text = 'هذا مثال عربي اصطناعي لاختبار معالجة النصوص.'
probs = classify(text)
print(probs)

# Training Setup

## Dataset Prep

### AI

In [ ]:
# import pandas as pd
# import numpy as np

# ai_df = pd.read_csv('../Tweets/AIArabicTweets.csv')
# ai_df.columns = ['text', 'label']
# # set all the labels to 0
# ai_df['label'] = 0
# ai_df = ai_df.drop(0)

# human_df = pd.read_csv('../Tweets/HumanArabicTweets.csv')
# human_df.columns = ['text', 'label']
# # set all the labels to 1
# human_df['label'] = 1

# # merge the two dataframes
# df = pd.concat([ai_df, human_df], ignore_index=True)
# df = df.sample(frac=1).reset_index(drop=True)
# # drop nan values
# df = df.dropna()

# # save the dataframe to a csv file
# df.to_csv('ArabicTweets.csv', index=False)


## Data Preprocessing

In [ ]:
import pandas as pd
from datasets import Dataset

# Load your CSV file with the correct encoding
df = pd.read_csv(r"data/input.csv", encoding="windows-1256")

# Convert the DataFrame into a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split the dataset into train and test sets
dataset = dataset.train_test_split(test_size=0.2)




In [ ]:
print(dataset)

In [ ]:
dataset["train"]

In [ ]:
dataset["train"][0]

### Applying araBERT Preprocessor

In [ ]:
def preprocess_function(examples):
    arabic_prep = arabert_prep.preprocess(examples["text"])
    result = tokenizer(arabic_prep,truncation=True,   
                       max_length=512, return_overflowing_tokens=True)

    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result
    # return tokenizer(arabic_prep, truncation=True, max_length=512, padding="max_length")

In [ ]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Setup Evaluation

In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
from sklearn.metrics import f1_score, recall_score, accuracy_score
import numpy as np
def calculate_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc = accuracy.compute(predictions=predictions, references=labels)
    f1 = f1_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')

    return {"accuracy": acc, "f1": f1, "recall": recall}

## Training

In [ ]:
# find number of labels
num_labels = len(set(dataset["train"]["label"]))
print(num_labels)

# get label names
label_names = ["AI", "Human"]
print(label_names)

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

In [ ]:
print(id2label)
print(label2id)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id
)

In [ ]:
batch_size = 1
exp = "1"
training_args = TrainingArguments(
    output_dir="trained/araBERT-base"+"_exp"+exp,
    learning_rate=2e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=20,
    save_total_limit=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    fp16=False,
    logging_dir="logs/araBERT-base"+"_exp"+exp,
    logging_strategy = "epoch",
    logging_steps = 1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=calculate_metrics,
)

In [ ]:
# trainer.train()

## Final Inference

In [ ]:
from transformers import AutoTokenizer
model_path = r"trained/araBERT-base_exp1"
tokenizer = AutoTokenizer.from_pretrained(model_path)

text = dataset["train"][0]["text"]

inputs = tokenizer(text, return_tensors="pt")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs.to(device)

with torch.no_grad():
    logits = model(**inputs).logits

In [ ]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

### Evaluation

In [ ]:
from tqdm.notebook import tqdm

def evaluate_test_set(model, tokenizer, test_dataset, batch_size=32):
    if torch.cuda.is_available():
        device = "cuda"
    else :
        device = "cpu"
    print(f"Device: {device}")
    model.to(device)
    model.eval()
    predictions = []
    labels = []
    accuracy = 0
    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)
    progress = tqdm(enumerate(test_dataloader), total=len(test_dataloader))
    for idx, batch in enumerate(test_dataloader):
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=512)
        labels.extend(batch["label"])
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        scores = outputs.logits
        probs = scores.softmax(dim=1)
        predicted_class_ids = probs.argmax(dim=1).tolist()

        batch_accuracy = sum([1 if label == prediction else 0 for label, prediction in zip(batch["label"], predicted_class_ids)])
        batch_accuracy = batch_accuracy / len(batch["label"])

        accuracy += batch_accuracy

        predictions.extend(predicted_class_ids)

        progress.update(1)
        progress.set_description(f"Batch {idx+1}/{len(test_dataloader)}: {batch_accuracy*100:.2f}%")
        # print(f"Batch {idx+1}/{len(test_dataloader)}: {batch_accuracy}")


    accuracy = accuracy / len(test_dataloader)
    return accuracy, labels, predictions



In [ ]:
for example in dataset["test"]:
    print(example["text"])
    print(example["label"])
    break

In [ ]:
accuracy, labels, predictions =  evaluate_test_set(model, tokenizer, dataset["test"], batch_size=256)

In [ ]:
accuracy, labels, predictions =  evaluate_test_set(model, tokenizer, dataset["test"], batch_size=256)

In [ ]:
print(f"Final accuracy: {accuracy*100}%")

In [ ]:
from tqdm.notebook import tqdm

# create a progress bar
progress = tqdm(total=len(dataset["test"]))
# set the model to evaluation mode
